<a href="https://colab.research.google.com/github/ibtihal7alharbi-tech/medical-cost-prediction/blob/main/Banan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
from sklearn.metrics import mean_absolute_error # Import mean_absolute_error

In [2]:
df = pd.read_csv("/content/insurance.csv")
df
#Charges col is the target

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [3]:
df.drop_duplicates(inplace=True)
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [4]:
# Text into numbers
le = LabelEncoder()

df['sex'] = le.fit_transform(df['sex'])       # male/female -> 1/0
df['smoker'] = le.fit_transform(df['smoker']) # yes/no -> 1/0
df['region'] = le.fit_transform(df['region']) # 4 regions -> 0,1,2,3

# drop the target col from X
X = df.drop('charges', axis=1)
y = df['charges']

df

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.92400
1,18,1,33.770,1,0,2,1725.55230
2,28,1,33.000,3,0,2,4449.46200
3,33,1,22.705,0,0,1,21984.47061
4,32,1,28.880,0,0,1,3866.85520
...,...,...,...,...,...,...,...
1333,50,1,30.970,3,0,1,10600.54830
1334,18,0,31.920,0,0,0,2205.98080
1335,18,0,36.850,0,0,2,1629.83350
1336,21,0,25.800,0,0,3,2007.94500


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_x.fit_transform(X_train)
X_test = scaler_x.transform(X_test)

y_train_scaled = scaler_y.fit_transform(np.array(y_train).reshape(-1, 1)).astype('float32')
y_test_scaled = scaler_y.transform(np.array(y_test).reshape(-1, 1)).astype('float32')

In [6]:
def build_baseline_model(lr):
    model = keras.Sequential([
        keras.layers.Dense(128, activation="relu", input_shape=(6,)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1) # Changed to 1 neuron with linear activation for regression
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=["mae"]
    )
    return model

In [7]:
learning_rates = [0.1, 0.01, 0.001, 0.0001]

for lr in learning_rates:
    print(f"\nTraining with LR = {lr}")
    model = build_baseline_model(lr)
    model.fit(X_train, y_train_scaled, epochs=100, batch_size=32, verbose=0)
    loss, mae = model.evaluate(X_test, y_test_scaled, verbose=0)
    print(f"LR {lr} → mae: {mae:.4f}")


Training with LR = 0.1


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


LR 0.1 → mae: 0.2850

Training with LR = 0.01
LR 0.01 → mae: 0.2452

Training with LR = 0.001
LR 0.001 → mae: 0.2522

Training with LR = 0.0001
LR 0.0001 → mae: 0.2317


In [8]:
y_pred_scaled = model.predict(X_test)

y_pred_final = scaler_y.inverse_transform(y_pred_scaled).flatten()
y_true_final = y_test.values.flatten()

# $$$
mae_dollars = mean_absolute_error(y_true_final, y_pred_final)

#  MAPE
mape = np.mean(np.abs((y_true_final - y_pred_final) / y_true_final)) * 100

#  (Accuracy)
accuracy = 100 - mape

#
print("="*40)
print(f" (MAPE): {mape:.2f} %")
print(f"  (Accuracy): {accuracy:.2f} %")
print("="*40)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
 (MAPE): 30.33 %
  (Accuracy): 69.67 %


In [9]:
def evaluate_model_full(model, X_test_data, y_test_data, scaler_y_obj):
    y_pred_scaled = model.predict(X_test_data, verbose=0)

    y_pred_final = scaler_y_obj.inverse_transform(y_pred_scaled).flatten()
    y_true_final = y_test_data.values.flatten()


    mae_dollars = mean_absolute_error(y_true_final, y_pred_final)
    mape = np.mean(np.abs((y_true_final - y_pred_final) / y_true_final)) * 100
    accuracy = 100 - mape

    print("="*40)
    print(f"(MAE): {mae_dollars:.2f} $")
    print(f"(MAPE): {mape:.2f} %")
    print(f"(Accuracy): {accuracy:.2f} %")
    print("="*40)

In [10]:
#Tuning
def build_tuned_model():
    model = Sequential([
        Dense(256, activation='relu', input_shape=(6,)),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

tuned_model = build_tuned_model()
tuned_model.fit(X_train, y_train_scaled, epochs=100, batch_size=32, verbose=0)

print("Tuning:")
evaluate_model_full(tuned_model, X_test, y_test, scaler_y)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Tuning:
(MAE): 3282.48 $
(MAPE): 46.31 %
(Accuracy): 53.69 %


In [11]:
#Dropout without tuning
def build_dropout_model():
    model = Sequential([
        Dense(128, activation='relu', input_shape=(6,)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=["mae"]
    )
    return model

dropout_model = build_dropout_model()
dropout_model.fit(X_train, y_train_scaled, epochs=100, batch_size=32, verbose=0)

print("Dropout:")
evaluate_model_full(dropout_model, X_test, y_test, scaler_y)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Dropout:
(MAE): 3056.54 $
(MAPE): 37.83 %
(Accuracy): 62.17 %


In [12]:
#dropout with tuning
def build_dropout1_model():
    model = Sequential([
        Dense(256, activation='relu', input_shape=(6,)),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

dropout1_model = build_dropout1_model()
dropout1_model.fit(X_train, y_train_scaled, epochs=100, batch_size=32, verbose=0)

print("Dropout after tuning:")
evaluate_model_full(dropout1_model, X_test, y_test, scaler_y)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Dropout after tuning:
(MAE): 2553.76 $
(MAPE): 30.40 %
(Accuracy): 69.60 %


In [13]:
#L2:
from tensorflow.keras import regularizers

def build_l2_model():
    model = Sequential([
        Dense(256, activation='relu', input_shape=(6,), kernel_regularizer=regularizers.l2(0.01)),
        Dropout(0.3),
        Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

l2_model = build_l2_model()
l2_model.fit(X_train, y_train_scaled, epochs=100, batch_size=32, verbose=0)


print("L2:")
evaluate_model_full(l2_model, X_test, y_test, scaler_y)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


L2:
(MAE): 2826.72 $
(MAPE): 36.16 %
(Accuracy): 63.84 %


In [14]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)
def build_final_early_model():
    model = Sequential([
        Dense(256, activation='relu', input_shape=(6,), kernel_regularizer=regularizers.l2(0.01)),
        Dropout(0.3),
        Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

final_model_es = build_final_early_model()

print("Training with Early Stopping...")
final_model_es.fit(
    X_train, y_train_scaled,
    validation_data=(X_test, y_test_scaled), # ضروري عشان يراقب الـ val_loss
    epochs=1000,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print("\n⭐ نتائج الموديل مع Early Stopping:")
evaluate_model_full(final_model_es, X_test, y_test, scaler_y)

Training with Early Stopping...
Epoch 1/1000


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 1.9741 - mae: 0.4476 - val_loss: 1.4561 - val_mae: 0.2792
Epoch 2/1000
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.2480 - mae: 0.3018 - val_loss: 0.9933 - val_mae: 0.2598
Epoch 3/1000
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.8961 - mae: 0.2922 - val_loss: 0.7268 - val_mae: 0.2662
Epoch 4/1000
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6774 - mae: 0.2820 - val_loss: 0.5845 - val_mae: 0.2506
Epoch 5/1000
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.5493 - mae: 0.2809 - val_loss: 0.4678 - val_mae: 0.2870
Epoch 6/1000
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4547 - mae: 0.2855 - val_loss: 0.3864 - val_mae: 0.2321
Epoch 7/1000
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4001 - mae: 0.2701 - val_loss: 0.3478 - val_mae: 0.2862
Epoch 8/1000
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3496 - mae: 0.2744 - val_loss: 0.3098 - val_mae: 0.2604
Epoch 9/1000
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3178 -